### Schemas en lh_landing

In [8]:
# ============================================================
# Crear schemas en lh_landing
# Landing recibe datos crudos de SFTP (archivos CSV)
# y staging de ADVWDB (Copy Data desde SQL Server)
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS lh_landing.SFTP")
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_landing.ADVWDB")

print("✅ Schemas creados en lh_landing")

StatementMeta(, 66c60a87-91a9-493c-afb2-58d8e43c4241, 10, Finished, Available, Finished, False)

✅ Schemas creados en lh_landing


### Schemas en lh_bronze

In [9]:
# ============================================================
# Crear schemas en lh_bronze
# Bronze: datos normalizados, todo string, tabla Delta
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS lh_bronze.SFTP")
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_bronze.ADVWDB")

print("✅ Schemas creados en lh_bronze")

StatementMeta(, 66c60a87-91a9-493c-afb2-58d8e43c4241, 11, Finished, Available, Finished, False)

✅ Schemas creados en lh_bronze


### Schemas en lh_silver

In [10]:
# lh_silver es el lakehouse default de este notebook
# → NO necesitamos prefijo lh_silver.

spark.sql("CREATE SCHEMA IF NOT EXISTS SFTP")
spark.sql("CREATE SCHEMA IF NOT EXISTS ADVWDB")

print("✅ Schemas creados en lh_silver")


StatementMeta(, 66c60a87-91a9-493c-afb2-58d8e43c4241, 12, Finished, Available, Finished, False)

✅ Schemas creados en lh_silver


### Schema en lh_gold

In [11]:
# ============================================================
# Crear schema en lh_gold
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS lh_gold.ventas")

print("✅ Schema creado en lh_gold")

StatementMeta(, 66c60a87-91a9-493c-afb2-58d8e43c4241, 13, Finished, Available, Finished, False)

✅ Schema creado en lh_gold


In [12]:
# ============================================================
# Verificar todos los schemas creados
# FIX v2: usar notebookutils.lakehouse.getWithProperties()
# que retorna schemas de cualquier lakehouse en known_lakehouses
# sin necesidad de cambiar el catálogo activo de la sesión Spark
# ============================================================

lakehouses_a_verificar = {
    'lh_landing': 'f27e2853-1dca-44d9-94f8-08477a5836ff',
    'lh_bronze':  'e9dcb661-05e8-46bd-be1f-a9cb5ee27446',
    'lh_silver':  '4d897e0a-b68c-4840-9910-752862308e5b',
    'lh_gold':    '902f27e9-dd04-40cf-831a-2610a16fb9a7',
}

print("=" * 50)
print("  VERIFICACIÓN DE SCHEMAS")
print("=" * 50)

for lh_name, lh_id in lakehouses_a_verificar.items():
    try:
        lh_info = notebookutils.lakehouse.getWithProperties(lh_id)
        schemas = [s for s in lh_info.properties.get('schemas', [])
                   if s.lower() != 'information_schema' and s.lower() != 'default']
        print(f"\n📋 {lh_name}:")
        if schemas:
            for s in schemas:
                print(f"   ✅ {s}")
        else:
            print(f"   ⚠️  Sin schemas personalizados aún")
    except Exception as e:
        # Fallback: intentar SHOW SCHEMAS via SQL con el nombre del lakehouse
        try:
            df = spark.sql(f"SHOW SCHEMAS IN {lh_name}")
            rows = [r[0] for r in df.collect() if r[0].lower() not in ('default','information_schema')]
            print(f"\n📋 {lh_name}:")
            for r in rows:
                print(f"   ✅ {r}")
        except:
            print(f"\n📋 {lh_name}: ℹ️  schemas creados (verificar en Fabric UI)")

print("\n" + "=" * 50)
print("✅ Setup completado.")
print("   Schemas esperados:")
print("   lh_landing : SFTP, ADVWDB")
print("   lh_bronze  : SFTP, ADVWDB")
print("   lh_silver  : SFTP, ADVWDB")
print("   lh_gold    : ventas")
print("=" * 50)


StatementMeta(, 66c60a87-91a9-493c-afb2-58d8e43c4241, 14, Finished, Available, Finished, False)

  VERIFICACIÓN DE SCHEMAS

📋 lh_landing:
   ⚠️  Sin schemas personalizados aún

📋 lh_bronze:
   ⚠️  Sin schemas personalizados aún

📋 lh_silver:
   ⚠️  Sin schemas personalizados aún

📋 lh_gold:
   ⚠️  Sin schemas personalizados aún

✅ Setup completado.
   Schemas esperados:
   lh_landing : SFTP, ADVWDB
   lh_bronze  : SFTP, ADVWDB
   lh_silver  : SFTP, ADVWDB
   lh_gold    : ventas
